# Lab M5 · Construye un harness desde cero

> **[ANCLA · M5]** Este lab es uno de los pilares de M5 (Harness desde cero). Hazlo entero.

**KEEPCODING · AI Engineering · Clase 5 · Módulo M5 "Harness Engineering"**

En este laboratorio vas a construir, en unas 150-200 líneas de Python, un **harness** completo: la capa de ingeniería que rodea al LLM y lo convierte en algo útil. La tesis del módulo es directa: **el modelo es el 20%, el harness es el 80%**.

## Qué necesitas

- Este notebook está pensado para ejecutarse en **Google Colab** de arriba a abajo.
- Necesitas una clave de OpenAI en la variable de entorno `OPENAI_API_KEY`.
  - En Colab: panel izquierdo, icono de la llave (Secrets), añade un secreto llamado `OPENAI_API_KEY` y activa "Notebook access".
  - El notebook también acepta introducir la clave a mano si no usas Secrets.
- Modelo usado: `gpt-5-mini`.
- Conexión a internet: la tool consulta la API pública de One Piece, con fallback local si falla.

## El hilo conductor

Usaremos una sola pregunta como termómetro durante todo el lab:

> *¿Cuál es la recompensa actual de Monkey D. Luffy?*

Primero se la haremos al **LLM desnudo** (sin herramientas). Luego construiremos el harness pieza a pieza y le haremos la **misma pregunta**. Mismo modelo, resultado muy distinto.

## Objetivos de aprendizaje

Al terminar sabrás construir y explicar los **6 pilares** de un harness:

1. **Tool registry** · registrar funciones como herramientas con su schema.
2. **Context builder** · ensamblar el prompt que ve el modelo.
3. **El loop** · el ciclo percepción, decisión, acción, observación.
4. **Memoria** · persistir lo que ocurre.
5. **Sandbox / allowlist** · controlar qué se puede ejecutar.
6. **La integración** · todo junto resolviendo la tarea real.

## Outline

1. Setup: instalar el SDK y configurar la API key.
2. El LLM desnudo: el "antes".
3. Pilar 1 · Tool registry con un decorador `@tool`.
4. Pilar 2 · El context builder.
5. Pilar 3 · El loop agéntico.
6. Pilar 4 · Memoria append-only.
7. Pilar 5 · Sandbox de allowlist.
8. La demo: la misma pregunta por el harness completo.
9. Recap y retos.

## 1. Setup

Instalamos el SDK oficial de OpenAI. En Colab esta celda tarda unos segundos.

El SDK nos da el cliente que llama al modelo. Todo lo demás (registry, loop, memoria, sandbox) lo escribimos nosotros: eso **es** el harness.

In [1]:
# Instalamos el SDK oficial de OpenAI (en Colab)
%pip install -q "openai>=1.50.0"

In [2]:
# Imports y configuración de la API key
from __future__ import annotations

import os
import json
import inspect
from typing import Any, Callable
from urllib.request import urlopen, Request
from urllib.parse import quote
from urllib.error import URLError, HTTPError

from openai import OpenAI

# Obtener la clave: primero Colab Secrets, si no, pedirla a mano
if not os.environ.get("OPENAI_API_KEY"):
    try:
        from google.colab import userdata  # type: ignore
        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    except Exception:
        import getpass
        os.environ["OPENAI_API_KEY"] = getpass.getpass("Introduce tu OPENAI_API_KEY: ")

# El modelo es fijo durante todo el lab: lo que cambia es el harness
MODEL = "gpt-5-mini"

# Un único cliente reutilizable
client = OpenAI()

print("SDK listo. Modelo:", MODEL)

SDK listo. Modelo: gpt-5-mini


## 2. El LLM desnudo · el "antes"

Antes de construir nada, hagamos la pregunta del lab directamente al modelo, sin herramientas ni contexto extra. Esto es un LLM **desnudo**.

El modelo tiene una fecha de corte de conocimiento y no consulta nada en vivo. Cuando le pedimos un dato actual y concreto (una recompensa que cambia con cada arco de la saga), pasa una de dos cosas:

- **Alucina**: inventa una cifra que suena plausible.
- **Se rinde**: dice que no puede saber el dato actual.

Ninguna de las dos sirve. Observa la respuesta: ese es el problema que el harness va a resolver.

In [3]:
# La pregunta que usaremos durante todo el lab
PREGUNTA = "¿Cuál es la recompensa actual de Monkey D. Luffy?"

# Llamada directa al LLM: sin tools, sin harness
respuesta_desnuda = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "Eres un asistente experto en One Piece."},
        {"role": "user", "content": PREGUNTA},
    ],
)

print("PREGUNTA:", PREGUNTA)
print("-" * 60)
print("LLM DESNUDO responde:")
print(respuesta_desnuda.choices[0].message.content)

PREGUNTA: ¿Cuál es la recompensa actual de Monkey D. Luffy?
------------------------------------------------------------
LLM DESNUDO responde:
La recompensa actual de Monkey D. Luffy es de 3.000.000.000 Berries.  
(Esta cifra fue publicada por el Gobierno Mundial tras los sucesos del arco Egghead, en el manga hasta el capítulo 1053.)


## 3. Pilar 1 · Tool registry

El modelo no puede consultar internet, pero **sí puede pedirnos que lo hagamos por él**. Para eso necesita saber qué herramientas existen. Una herramienta es: una función de Python + una descripción en JSON que el modelo entiende.

Escribir ese JSON a mano para cada función es tedioso y se desincroniza con el código. Solución: un decorador `@tool` que, usando el módulo `inspect`, lee la **firma** de la función y construye el JSON schema automáticamente.

Así, registrar una herramienta es solo poner `@tool` encima de una función bien anotada. El registry guarda dos cosas por cada tool: la función ejecutable y su schema.

In [4]:
# El registry: nombre de tool -> {"func": callable, "schema": dict}
TOOL_REGISTRY: dict[str, dict[str, Any]] = {}

# Mapeo de tipos de Python al tipo JSON Schema que espera el modelo
_JSON_TYPES = {str: "string", int: "integer", float: "number", bool: "boolean"}


def tool(func: Callable) -> Callable:
    """Decorador: registra la función y construye su JSON schema desde la firma."""
    sig = inspect.signature(func, eval_str=True)
    properties: dict[str, Any] = {}
    required: list[str] = []

    for name, param in sig.parameters.items():
        py_type = param.annotation if param.annotation is not inspect.Parameter.empty else str
        properties[name] = {"type": _JSON_TYPES.get(py_type, "string")}
        if param.default is inspect.Parameter.empty:
            required.append(name)

    # La descripción sale del docstring de la función
    schema = {
        "type": "function",
        "function": {
            "name": func.__name__,
            "description": (func.__doc__ or "").strip(),
            "parameters": {
                "type": "object",
                "properties": properties,
                "required": required,
            },
        },
    }

    TOOL_REGISTRY[func.__name__] = {"func": func, "schema": schema}
    return func


print("Decorador @tool listo. Registry vacío:", list(TOOL_REGISTRY))

Decorador @tool listo. Registry vacío: []


Ahora registramos nuestra herramienta real: `buscar_personaje`. Consulta la **API pública de One Piece** (`api-onepiece.com`) y, si la red falla o el personaje no aparece, recurre a un **diccionario local** de personajes y recompensas conocidas.

Ese fallback no es un detalle menor: en una clase no podemos depender de que una API pública esté siempre disponible. Un harness robusto **degrada con elegancia** en vez de romperse.

In [5]:
import unicodedata

# Diccionario local de respaldo: personaje -> datos conocidos
FALLBACK_PERSONAJES: dict[str, dict[str, Any]] = {
    "monkey d. luffy": {"nombre": "Monkey D. Luffy", "recompensa": "3.000.000.000 berries", "tripulacion": "Piratas de Sombrero de Paja"},
    "roronoa zoro": {"nombre": "Roronoa Zoro", "recompensa": "1.111.000.000 berries", "tripulacion": "Piratas de Sombrero de Paja"},
    "nami": {"nombre": "Nami", "recompensa": "366.000.000 berries", "tripulacion": "Piratas de Sombrero de Paja"},
    "sanji": {"nombre": "Sanji", "recompensa": "1.032.000.000 berries", "tripulacion": "Piratas de Sombrero de Paja"},
    "trafalgar law": {"nombre": "Trafalgar D. Water Law", "recompensa": "3.000.000.000 berries", "tripulacion": "Piratas Heart"},
    "portgas d. ace": {"nombre": "Portgas D. Ace", "recompensa": "550.000.000 berries", "tripulacion": "Piratas de Barbablanca"},
}

# Cache en memoria: la lista completa de personajes se descarga una sola vez.
_OP_CACHE: list[dict[str, Any]] = []


def _normalizar(texto: str) -> str:
    """Minúsculas, sin acentos ni puntuación, espacios colapsados."""
    t = unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode()
    t = "".join(c if c.isalnum() else " " for c in t.lower())
    return " ".join(t.split())


def _coincide(consulta: str, nombre: str) -> bool:
    """True si 'consulta' identifica a 'nombre'. Tolera puntos, acentos y orden.

    El LLM puede pasar "Luffy" o "Monkey D. Luffy"; la API guarda
    "Monkey D Luffy" (sin punto). Normalizamos ambos y comparamos por
    contención o por palabras, nunca con un substring frágil.
    """
    q, n = _normalizar(consulta), _normalizar(nombre)
    if not q or not n:
        return False
    if q == n or q in n or n in q:
        return True
    palabras = [w for w in q.split() if len(w) >= 3]
    return bool(palabras) and all(w in n.split() for w in palabras)


def _cargar_personajes_op() -> list[dict[str, Any]]:
    """Descarga (una sola vez) la lista completa de personajes de One Piece."""
    global _OP_CACHE
    if _OP_CACHE:
        return _OP_CACHE
    url = "https://api.api-onepiece.com/v2/characters/en"
    req = Request(url, headers={"User-Agent": "keepcoding-harness-lab/1.0"})
    with urlopen(req, timeout=25) as resp:
        _OP_CACHE = json.loads(resp.read().decode("utf-8"))
    return _OP_CACHE


def _buscar_en_api(nombre: str) -> dict[str, Any] | None:
    """Intenta la API pública de One Piece. Devuelve None si falla o no hay datos."""
    try:
        for p in _cargar_personajes_op():
            if _coincide(nombre, p.get("name") or ""):
                return {
                    "nombre": p.get("name"),
                    "recompensa": p.get("bounty") or "desconocida",
                    "tripulacion": (p.get("crew") or {}).get("name", "desconocida"),
                    "fuente": "API api-onepiece.com",
                }
        return None
    except (URLError, HTTPError, TimeoutError, ValueError, KeyError):
        return None


@tool
def buscar_personaje(nombre: str) -> dict[str, Any]:
    """Busca un personaje de One Piece y devuelve su recompensa (bounty) y tripulación. Usa el parámetro 'nombre'."""
    # 1) Intento principal: API en vivo
    resultado = _buscar_en_api(nombre)
    if resultado is not None:
        return resultado
    # 2) Fallback: diccionario local
    for clave, datos in FALLBACK_PERSONAJES.items():
        if _coincide(nombre, clave) or _coincide(nombre, datos["nombre"]):
            return {**datos, "fuente": "diccionario local de respaldo"}
    # 3) No encontrado en ninguna fuente
    return {"error": f"No se encontró información sobre '{nombre}'."}


print("Tool registrada:", list(TOOL_REGISTRY))

Tool registrada: ['buscar_personaje']


In [6]:
# Comprobamos el schema que el decorador generó solo
print("Schema autogenerado para buscar_personaje:")
print(json.dumps(TOOL_REGISTRY["buscar_personaje"]["schema"], indent=2, ensure_ascii=False))

print("\nPrueba directa de la tool (sin LLM todavía):")
print(buscar_personaje("Monkey D. Luffy"))

Schema autogenerado para buscar_personaje:
{
  "type": "function",
  "function": {
    "name": "buscar_personaje",
    "description": "Busca un personaje de One Piece y devuelve su recompensa (bounty) y tripulación. Usa el parámetro 'nombre'.",
    "parameters": {
      "type": "object",
      "properties": {
        "nombre": {
          "type": "string"
        }
      },
      "required": [
        "nombre"
      ]
    }
  }
}

Prueba directa de la tool (sin LLM todavía):


{'nombre': 'Monkey D Luffy', 'recompensa': '3.000.000.000', 'tripulacion': 'The Chapeau de Paille crew', 'fuente': 'API api-onepiece.com'}


## 4. Pilar 2 · El context builder

El modelo solo ve lo que le pasamos en `messages`. El **context builder** es la pieza que ensambla ese contexto de forma ordenada y repetible:

- El **system prompt**: rol, instrucciones y cómo debe comportarse.
- El **historial**: los mensajes intercambiados hasta ahora.

Las descripciones de las tools no van en el texto: viajan aparte, en el parámetro `tools` de la API. Aun así, el context builder es quien decide qué historial entra. Lo encapsulamos en una función para no repetir código y para tener un único sitio donde tocar el prompt.

In [7]:
# System prompt: el rol y las reglas de comportamiento del agente
SYSTEM_PROMPT = (
    "Eres un asistente experto en One Piece. "
    "Cuando te pregunten por datos concretos de un personaje (recompensa, tripulación), "
    "NO respondas de memoria: usa la herramienta buscar_personaje. "
    "Después de obtener el resultado, responde en español, claro y conciso, "
    "e indica de qué fuente proviene el dato. "
    "Formato: texto plano, sin markdown ni rayas largas."
)


def construir_contexto(historial: list[dict[str, Any]]) -> list[dict[str, Any]]:
    """Ensambla los mensajes que verá el modelo: system prompt + historial."""
    return [{"role": "system", "content": SYSTEM_PROMPT}] + historial


def schemas_de_tools() -> list[dict[str, Any]]:
    """Devuelve la lista de schemas para el parámetro 'tools' de la API."""
    return [entry["schema"] for entry in TOOL_REGISTRY.values()]


# Demostración: contexto para una conversación que empieza
demo_ctx = construir_contexto([{"role": "user", "content": PREGUNTA}])
for m in demo_ctx:
    print(f"[{m['role']}] {m['content'][:90]}...")

[system] Eres un asistente experto en One Piece. Cuando te pregunten por datos concretos de un pers...
[user] ¿Cuál es la recompensa actual de Monkey D. Luffy?...


## 5. Pilar 3 · El loop agéntico

Aquí está el corazón del harness. El modelo no ejecuta nada: solo **decide**. Nosotros ejecutamos. El ciclo se repite hasta que el modelo da una respuesta final:

1. **Percepción**: enviamos el contexto al modelo.
2. **Decisión**: el modelo responde con texto final, o con una **tool call**.
3. **Acción**: si pidió una tool, la ejecutamos nosotros.
4. **Observación**: devolvemos el resultado al modelo como un mensaje de rol `tool`.
5. Volvemos al paso 1.

El parámetro `tool_calls` de la respuesta nos dice qué quiere el modelo. Cada tool call lleva un `id`, un `name` y unos `arguments` en JSON. El mensaje `tool` que devolvemos debe referenciar ese mismo `tool_call_id`.

Ponemos un límite de iteraciones (`max_iteraciones`): un harness nunca debe permitir bucles infinitos. La función usa `ejecutar_tool`, que definiremos en el pilar 5 (el sandbox); como el loop solo la **llama** en tiempo de ejecución, definirla después es correcto: el notebook se ejecuta de arriba a abajo y el loop no corre hasta la demo.

In [8]:
# El loop agéntico. Recibe el historial y lo va ampliando con cada paso.
# Usa ejecutar_tool(), que definiremos en el pilar del sandbox (pilar 5).

def agente_loop(historial: list[dict[str, Any]], max_iteraciones: int = 5) -> str:
    """Ejecuta el ciclo percepción-decisión-acción-observación hasta la respuesta final."""
    for i in range(1, max_iteraciones + 1):
        # 1) PERCEPCIÓN: el modelo ve el contexto completo
        respuesta = client.chat.completions.create(
            model=MODEL,
            messages=construir_contexto(historial),
            tools=schemas_de_tools(),
        )
        mensaje = respuesta.choices[0].message

        # 2) DECISIÓN: sin tool calls => es la respuesta final
        if not mensaje.tool_calls:
            print(f"[iter {i}] El modelo responde (final).")
            historial.append({"role": "assistant", "content": mensaje.content})
            return mensaje.content or ""

        # El mensaje del assistant que pide tools debe ir al historial tal cual
        historial.append(mensaje.model_dump(exclude_none=True))

        # 3) ACCIÓN + 4) OBSERVACIÓN: ejecutamos cada tool call pedida
        for tc in mensaje.tool_calls:
            nombre = tc.function.name
            argumentos = json.loads(tc.function.arguments or "{}")
            print(f"[iter {i}] El modelo pide la tool: {nombre}({argumentos})")

            resultado = ejecutar_tool(nombre, argumentos)  # pasa por el sandbox

            historial.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": json.dumps(resultado, ensure_ascii=False),
            })

    return "Límite de iteraciones alcanzado sin respuesta final."


print("agente_loop definido.")

agente_loop definido.


## 6. Pilar 4 · Memoria append-only

El historial vive en memoria mientras corre el loop, pero un harness serio necesita **registrar lo que pasó**: para depurar, auditar y entender por qué el agente decidió lo que decidió.

Usamos la forma más simple y honesta de memoria: **append-only**. Cada evento se añade al final, nada se borra ni se reescribe. Lo guardamos en una lista en memoria y, además, en un fichero **JSONL** (una línea JSON por evento), un formato cómodo para inspeccionar después.

Esto es una traza de ejecución. Es la base de lo que en producción se convierte en observabilidad.

In [9]:
# Memoria append-only: lista en memoria + fichero JSONL
from datetime import datetime, timezone
from pathlib import Path

MEMORIA: list[dict[str, Any]] = []
MEMORIA_PATH = "harness_memoria.jsonl"

# Cada ejecución del lab empieza con una traza limpia (dentro de ella, solo se añade)
Path(MEMORIA_PATH).unlink(missing_ok=True)


def recordar(tipo: str, datos: dict[str, Any]) -> None:
    """Añade un evento a la memoria. Nunca borra ni modifica eventos previos."""
    evento = {
        "ts": datetime.now(timezone.utc).isoformat(),
        "tipo": tipo,
        "datos": datos,
    }
    MEMORIA.append(evento)
    with open(MEMORIA_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(evento, ensure_ascii=False) + "\n")


# Prueba rápida
recordar("test", {"mensaje": "memoria inicializada"})
print("Eventos en memoria:", len(MEMORIA))
print("Último evento:", MEMORIA[-1])

Eventos en memoria: 1
Último evento: {'ts': '2026-10-19T13:10:30.590880+00:00', 'tipo': 'test', 'datos': {'mensaje': 'memoria inicializada'}}


## 7. Pilar 5 · El sandbox de allowlist

El modelo decide qué tool llamar, pero **no debe poder ejecutar lo que quiera**. Entre la decisión del modelo y la ejecución real ponemos un **guard**: una función que solo deja pasar tools que estén en una **allowlist** explícita.

Es el principio de mínimo privilegio. Cualquier intento de ejecutar algo fuera de la lista se **bloquea** y se registra en memoria, en vez de ejecutarse.

Definimos aquí `ejecutar_tool`, la función que el loop ya invocaba. Cada ejecución (permitida o bloqueada) deja su rastro en la memoria del pilar anterior. Para verlo en acción, forzaremos una llamada a una tool que no está permitida.

In [10]:
# Allowlist: SOLO estas tools pueden ejecutarse, pase lo que pase
ALLOWLIST: set[str] = {"buscar_personaje"}


def ejecutar_tool(nombre: str, argumentos: dict[str, Any]) -> dict[str, Any]:
    """Guard del sandbox: ejecuta una tool solo si está en la allowlist."""
    # Guard 1: la tool debe existir en el registry
    if nombre not in TOOL_REGISTRY:
        resultado = {"error": f"La tool '{nombre}' no existe en el registry."}
        recordar("tool_error", {"tool": nombre, "motivo": "no registrada"})
        return resultado

    # Guard 2: la tool debe estar explícitamente permitida
    if nombre not in ALLOWLIST:
        resultado = {"error": f"BLOQUEADO: la tool '{nombre}' no está en la allowlist."}
        recordar("tool_bloqueada", {"tool": nombre, "argumentos": argumentos})
        print(f"  [SANDBOX] Ejecución de '{nombre}' BLOQUEADA.")
        return resultado

    # Permitida: ejecutamos y registramos
    func = TOOL_REGISTRY[nombre]["func"]
    resultado = func(**argumentos)
    recordar("tool_ejecutada", {"tool": nombre, "argumentos": argumentos, "resultado": resultado})
    print(f"  [SANDBOX] Ejecución de '{nombre}' permitida.")
    return resultado


print("Sandbox listo. Allowlist:", ALLOWLIST)

Sandbox listo. Allowlist: {'buscar_personaje'}


In [11]:
# Demostración del guard: una llamada permitida y una bloqueada

# Registramos una tool peligrosa, pero NO la añadimos a la ALLOWLIST.
# Se queda en el registry a propósito: aunque el modelo la pida, el sandbox la bloqueará.
@tool
def borrar_base_de_datos(tabla: str) -> dict[str, Any]:
    """Borra una tabla de la base de datos (tool peligrosa de ejemplo)."""
    return {"borrada": tabla}

print("Caso 1 · tool en la allowlist:")
ok = ejecutar_tool("buscar_personaje", {"nombre": "Roronoa Zoro"})
print("  ->", ok)

print("\nCaso 2 · tool fuera de la allowlist (registrada, pero no permitida):")
bloqueada = ejecutar_tool("borrar_base_de_datos", {"tabla": "usuarios"})
print("  ->", bloqueada)

print("\nLa memoria registró ambos intentos:")
for ev in MEMORIA[-2:]:
    print(" ", ev["tipo"], "->", ev["datos"].get("tool"))

Caso 1 · tool en la allowlist:
  [SANDBOX] Ejecución de 'buscar_personaje' permitida.
  -> {'nombre': 'Roronoa Zoro', 'recompensa': '1.101.000.000', 'tripulacion': 'The Chapeau de Paille crew', 'fuente': 'API api-onepiece.com'}

Caso 2 · tool fuera de la allowlist (registrada, pero no permitida):
  [SANDBOX] Ejecución de 'borrar_base_de_datos' BLOQUEADA.
  -> {'error': "BLOQUEADO: la tool 'borrar_base_de_datos' no está en la allowlist."}

La memoria registró ambos intentos:
  tool_ejecutada -> buscar_personaje
  tool_bloqueada -> borrar_base_de_datos


## 8. La demo · la misma pregunta, ahora por el harness

Momento de la verdad. Vamos a lanzar **exactamente la misma pregunta** del paso 2 (el LLM desnudo), pero esta vez a través del harness completo: registry + context builder + loop + memoria + sandbox.

El modelo no cambió. Lo único que añadimos es la capa de ingeniería a su alrededor. Observa la traza: el modelo decidirá llamar a `buscar_personaje`, el sandbox la dejará pasar, la tool obtendrá la recompensa real (de la API o del fallback) y el modelo construirá la respuesta final con ese dato.

In [12]:
# La misma pregunta del paso 2, ahora por el harness completo
print("PREGUNTA:", PREGUNTA)
print("=" * 60)

historial = [{"role": "user", "content": PREGUNTA}]
recordar("pregunta_usuario", {"texto": PREGUNTA})

respuesta_harness = agente_loop(historial)
recordar("respuesta_final", {"texto": respuesta_harness})

print("=" * 60)
print("HARNESS responde:")
print(respuesta_harness)

PREGUNTA: ¿Cuál es la recompensa actual de Monkey D. Luffy?


[iter 1] El modelo pide la tool: buscar_personaje({'nombre': 'Monkey D. Luffy'})
  [SANDBOX] Ejecución de 'buscar_personaje' permitida.


[iter 2] El modelo responde (final).
HARNESS responde:
La recompensa actual de Monkey D. Luffy es de 3.000.000.000 Berries (fuente: API api-onepiece.com).


In [13]:
# Comparación lado a lado: el antes y el después
print("MISMO MODELO:", MODEL)
print("=" * 60)
print("ANTES · LLM desnudo (sin harness):")
print((respuesta_desnuda.choices[0].message.content or "").strip())
print("-" * 60)
print("DESPUÉS · LLM + harness:")
print((respuesta_harness or "").strip())
print("=" * 60)
print("La diferencia no es el modelo. Es el harness.")

MISMO MODELO: gpt-5-mini
ANTES · LLM desnudo (sin harness):
La recompensa actual de Monkey D. Luffy es de 3.000.000.000 Berries.  
(Esta cifra fue publicada por el Gobierno Mundial tras los sucesos del arco Egghead, en el manga hasta el capítulo 1053.)
------------------------------------------------------------
DESPUÉS · LLM + harness:
La recompensa actual de Monkey D. Luffy es de 3.000.000.000 Berries (fuente: API api-onepiece.com).
La diferencia no es el modelo. Es el harness.


### La traza completa en memoria

La memoria append-only guardó cada paso de la ejecución. Esta traza es lo que te permite responder "¿por qué el agente hizo esto?" sin adivinar.

In [14]:
# Recorremos la memoria: cada evento, en orden
print(f"Traza de ejecución · {len(MEMORIA)} eventos en {MEMORIA_PATH}")
print("-" * 60)
for i, ev in enumerate(MEMORIA, start=1):
    print(f"{i:>2}. [{ev['tipo']}]")
    for clave, valor in ev["datos"].items():
        texto = str(valor)
        if len(texto) > 80:
            texto = texto[:80] + "..."
        print(f"      {clave}: {texto}")

Traza de ejecución · 6 eventos en harness_memoria.jsonl
------------------------------------------------------------
 1. [test]
      mensaje: memoria inicializada
 2. [tool_ejecutada]
      tool: buscar_personaje
      argumentos: {'nombre': 'Roronoa Zoro'}
      resultado: {'nombre': 'Roronoa Zoro', 'recompensa': '1.101.000.000', 'tripulacion': 'The Ch...
 3. [tool_bloqueada]
      tool: borrar_base_de_datos
      argumentos: {'tabla': 'usuarios'}
 4. [pregunta_usuario]
      texto: ¿Cuál es la recompensa actual de Monkey D. Luffy?
 5. [tool_ejecutada]
      tool: buscar_personaje
      argumentos: {'nombre': 'Monkey D. Luffy'}
      resultado: {'nombre': 'Monkey D Luffy', 'recompensa': '3.000.000.000', 'tripulacion': 'The ...
 6. [respuesta_final]
      texto: La recompensa actual de Monkey D. Luffy es de 3.000.000.000 Berries (fuente: API...


## 9. Ejercicios

Refuerza lo aprendido tocando el harness que acabas de construir. La celda siguiente es un andamiaje con las soluciones.

1. **Nueva tool**: registra con `@tool` una función `comparar_recompensas(nombre_a, nombre_b)` que use `buscar_personaje` y diga quién tiene la recompensa mayor. Acuérdate de añadirla a la `ALLOWLIST`.
2. **Pregunta nueva**: lanza el harness con "¿quién tiene más recompensa, Zoro o Sanji?" y observa cuántas iteraciones hace el loop.
3. **Sandbox**: quita `buscar_personaje` de la `ALLOWLIST` y vuelve a lanzar la demo. ¿Qué responde ahora el harness y por qué?

**Pitfall común**: olvidar añadir la nueva tool a la `ALLOWLIST`. El modelo la pedirá (está en el registry y en los schemas), pero el sandbox la bloqueará. Es el comportamiento correcto: registry y allowlist son listas distintas a propósito.

In [15]:
# Andamiaje de soluciones a los ejercicios

# Ejercicio 1: una nueva tool que se apoya en la anterior
@tool
def comparar_recompensas(nombre_a: str, nombre_b: str) -> dict[str, Any]:
    """Compara las recompensas de dos personajes de One Piece y dice quién tiene la mayor."""
    a = buscar_personaje(nombre_a)
    b = buscar_personaje(nombre_b)
    return {"personaje_a": a, "personaje_b": b,
            "nota": "El modelo debe interpretar las cifras y decidir quién tiene más."}

# No olvides: el registry NO es la allowlist. Hay que permitirla explícitamente.
ALLOWLIST.add("comparar_recompensas")
print("Allowlist actualizada:", ALLOWLIST)

# Ejercicio 2: una pregunta nueva por el harness completo
hist_ej = [{"role": "user", "content": "¿Quién tiene más recompensa, Zoro o Sanji?"}]
print("\nRespuesta del harness al ejercicio 2:")
print(agente_loop(hist_ej))

# Ejercicio 3 (para probar tú mismo):
# ALLOWLIST.discard("buscar_personaje")  # descomenta y relanza la demo del paso 8

Allowlist actualizada: {'comparar_recompensas', 'buscar_personaje'}

Respuesta del harness al ejercicio 2:


[iter 1] El modelo pide la tool: buscar_personaje({'nombre': 'Zoro'})
  [SANDBOX] Ejecución de 'buscar_personaje' permitida.
[iter 1] El modelo pide la tool: buscar_personaje({'nombre': 'Sanji'})
  [SANDBOX] Ejecución de 'buscar_personaje' permitida.


[iter 2] El modelo responde (final).
Roronoa Zoro tiene la recompensa mayor: 1.101.000.000 (Berries) frente a Sanji, cuya recompensa es 1.032.000.000 (Berries). Ambos aparecen como miembros de "The Chapeau de Paille crew". Fuente: API api-onepiece.com.


## Recap · los 6 pilares y la tesis del 80%

Has construido un harness completo en unas 150-200 líneas. Repaso de las piezas:

| Pilar | Qué hace | En este lab |
|---|---|---|
| 1. Tool registry | Convierte funciones en herramientas con schema | decorador `@tool` + `inspect` |
| 2. Context builder | Ensambla lo que ve el modelo | `construir_contexto` + `SYSTEM_PROMPT` |
| 3. El loop | Ciclo percepción, decisión, acción, observación | `agente_loop` |
| 4. Memoria | Registra lo que pasa, append-only | `recordar` + JSONL |
| 5. Sandbox / allowlist | Controla qué se puede ejecutar | `ejecutar_tool` + `ALLOWLIST` |
| 6. Integración | Todo junto resolviendo la tarea | la demo del paso 8 |

### La tesis

Hicimos **dos veces la misma pregunta** al **mismo modelo** `gpt-5-mini`:

- **Sin harness**: alucina una cifra o se rinde.
- **Con harness**: llama a la tool, obtiene el bounty real y responde bien, citando la fuente.

El modelo no cambió ni una coma. Lo que cambió fue todo lo que lo rodea. **El modelo es el 20%. El harness es el 80%.** Ahí está el trabajo de un AI Engineer.

### Retos para seguir

- **Streaming**: muestra la respuesta token a token mientras se genera.
- **Más tools**: añade `buscar_tripulacion` o `buscar_fruta_del_diablo` usando otros endpoints de `api-onepiece.com`.
- **Reintentos**: envuelve la llamada al LLM en una política de reintento con backoff.
- **Recorte de contexto**: cuando el historial crezca, resume los mensajes antiguos en el context builder.
- **Tipos ricos en el schema**: extiende el decorador `@tool` para soportar `list`, `dict` o enums.
- **Evaluación**: crea un set de 10 preguntas y mide cuántas responde bien con harness frente a sin él.